# Étape 1 — Exploration et préparation des données
## Projet : Estimation du prix de vente de logements (House Prices)

**Objectif de ce notebook :**

- Analyser la structure et les types de données
- Identifier les valeurs manquantes et les doublons
- Rechercher les incohérences et valeurs aberrantes
- Distinguer variables numériques et catégorielles
- Analyser la variable cible `SalePrice`
- Traiter les valeurs manquantes
- Encoder les variables catégorielles
- Appliquer une normalisation / standardisation lorsque nécessaire
- Séparer `X` et `y`

⚠️ **Point d'attention central : éviter la fuite de données (data leakage).**
Toutes les statistiques utilisées pour le traitement (médianes, modes, moyennes, écarts-types,
catégories rencontrées...) seront calculées **uniquement sur l'ensemble d'entraînement**, puis
appliquées à l'ensemble de test à l'aide d'un `Pipeline` scikit-learn. C'est pourquoi la séparation
train/test est faite **avant** l'imputation, l'encodage et la mise à l'échelle.


## 0. Imports et configuration

In [2]:
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 150)


## 1. Chargement des données

Le fichier `House_Prices.csv` regroupe l'ensemble des logements avec leurs caractéristiques et
leur prix de vente (`SalePrice`), la variable cible du projet.


In [4]:
df = pd.read_csv('../data/raw/House_Prices.csv')
print(f"Dimensions du dataset : {df.shape[0]} lignes x {df.shape[1]} colonnes")
df.head()


Dimensions du dataset : 2919 lignes x 81 colonnes


,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,LotConfig,LandSlope,Neighborhood,Condition1,Condition2,BldgType,HouseStyle,OverallQual,OverallCond,YearBuilt,YearRemodAdd,RoofStyle,RoofMatl,Exterior1st,Exterior2nd,MasVnrType,MasVnrArea,ExterQual,ExterCond,Foundation,BsmtQual,BsmtCond,BsmtExposure,BsmtFinType1,BsmtFinSF1,BsmtFinType2,BsmtFinSF2,BsmtUnfSF,TotalBsmtSF,Heating,HeatingQC,CentralAir,Electrical,1stFlrSF,2ndFlrSF,LowQualFinSF,GrLivArea,BsmtFullBath,BsmtHalfBath,FullBath,HalfBath,BedroomAbvGr,KitchenAbvGr,KitchenQual,TotRmsAbvGrd,Functional,Fireplaces,FireplaceQu,GarageType,GarageYrBlt,GarageFinish,GarageCars,GarageArea,GarageQual,GarageCond,PavedDrive,WoodDeckSF,OpenPorchSF,EnclosedPorch,3SsnPorch,ScreenPorch,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,Inside,Gtl,CollgCr,Norm,Norm,1Fam,2Story,7,5,2003,2003,Gable,CompShg,VinylSd,VinylSd,BrkFace,196.0,Gd,TA,PConc,Gd,TA,No,GLQ,706.0,Unf,0.0,150.0,856.0,GasA,Ex,Y,SBrkr,856,854,0,1710,1.0,0.0,2,1,3,1,Gd,8,Typ,0,NaN,Attchd,2003.0,RFn,2.0,548.0,TA,TA,Y,0,61,0,0,0,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500.0
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,FR2,Gtl,Veenker,Feedr,Norm,1Fam,1Story,6,8,1976,1976,Gable,CompShg,MetalSd,MetalSd,NaN,0.0,TA,TA,CBlock,Gd,TA,Gd,ALQ,978.0,Unf,0.0,284.0,1262.0,GasA,Ex,Y,SBrkr,1262,0,0,1262,0.0,1.0,2,0,3,1,TA,6,Typ,1,TA,Attchd,1976.0,RFn,2.0,460.0,TA,TA,Y,298,0,0,0,0,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500.0
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,Inside,Gtl,CollgCr,Norm,Norm,1Fam,2Story,7,5,2001,2002,Gable,CompShg,VinylSd,VinylSd,BrkFace,162.0,Gd,TA,PConc,Gd,TA,Mn,GLQ,486.0,Unf,0.0,434.0,920.0,GasA,Ex,Y,SBrkr,920,866,0,1786,1.0,0.0,2,1,3,1,Gd,6,Typ,1,TA,Attchd,2001.0,RFn,2.0,608.0,TA,TA,Y,0,42,0,0,0,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500.0
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,Corner,Gtl,Crawfor,Norm,Norm,1Fam,2Story,7,5,1915,1970,Gable,CompShg,Wd Sdng,Wd Shng,NaN,0.0,TA,TA,BrkTil,TA,Gd,No,ALQ,216.0,Unf,0.0,540.0,756.0,GasA,Gd,Y,SBrkr,961,756,0,1717,1.0,0.0,1,0,3,1,Gd,7,Typ,1,Gd,Detchd,1998.0,Unf,3.0,642.0,TA,TA,Y,0,35,272,0,0,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000.0
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,FR2,Gtl,NoRidge,Norm,Norm,1Fam,2Story,8,5,2000,2000,Gable,CompShg,VinylSd,VinylSd,BrkFace,350.0,Gd,TA,PConc,Gd,TA,Av,GLQ,655.0,Unf,0.0,490.0,1145.0,GasA,Ex,Y,SBrkr,1145,1053,0,2198,1.0,0.0,2,1,4,1,Gd,9,Typ,1,TA,Attchd,2000.0,RFn,3.0,836.0,TA,TA,Y,192,84,0,0,0,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000.0


## 2. Analyse de la structure et des types de données

In [5]:
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 2919 entries, 0 to 2918
Data columns (total 81 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Id             2919 non-null   int64  
 1   MSSubClass     2919 non-null   int64  
 2   MSZoning       2915 non-null   str    
 3   LotFrontage    2433 non-null   float64
 4   LotArea        2919 non-null   int64  
 5   Street         2919 non-null   str    
 6   Alley          198 non-null    str    
 7   LotShape       2919 non-null   str    
 8   LandContour    2919 non-null   str    
 9   Utilities      2917 non-null   str    
 10  LotConfig      2919 non-null   str    
 11  LandSlope      2919 non-null   str    
 12  Neighborhood   2919 non-null   str    
 13  Condition1     2919 non-null   str    
 14  Condition2     2919 non-null   str    
 15  BldgType       2919 non-null   str    
 16  HouseStyle     2919 non-null   str    
 17  OverallQual    2919 non-null   int64  
 18  OverallCond    2919

In [6]:
# Répartition des types de données pandas
df.dtypes.value_counts()


str        43
int64      26
float64    12
Name: count, dtype: int64

In [7]:
# Statistiques descriptives des variables numériques
df.describe().T


,count,mean,std,min,25%,50%,75%,max
Id,2919.0,1460.000000,842.787043,1.0,730.500000,1460.000000,2189.500000,2919.0
MSSubClass,2919.0,57.137718,42.517628,20.0,20.000000,50.000000,70.000000,190.0
LotFrontage,2433.0,69.305795,23.344905,21.0,59.000000,68.000000,80.000000,313.0
LotArea,2919.0,10168.114080,7886.996359,1300.0,7478.000000,9453.000000,11570.000000,215245.0
OverallQual,2919.0,6.089072,1.409947,1.0,5.000000,6.000000,7.000000,10.0
OverallCond,2919.0,5.564577,1.113131,1.0,5.000000,5.000000,6.000000,9.0
YearBuilt,2919.0,1971.312778,30.291442,1872.0,1953.500000,1973.000000,2001.000000,2010.0
YearRemodAdd,2919.0,1984.264474,20.894344,1950.0,1965.000000,1993.000000,2004.000000,2010.0
MasVnrArea,2896.0,102.201312,179.334253,0.0,0.000000,0.000000,164.000000,1600.0
BsmtFinSF1,2918.0,441.423235,455.610826,0.0,0.000000,368.500000,733.000000,5644.0


In [10]:
# Aperçu des variables catégorielles (objets)
df.describe(include='object').T


C:\Users\ycode\AppData\Local\Temp\ipykernel_15756\1613851401.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  df.describe(include='object').T


,count,unique,top,freq
MSZoning,2915,5,RL,2265
Street,2919,2,Pave,2907
Alley,198,2,Grvl,120
LotShape,2919,4,Reg,1859
LandContour,2919,4,Lvl,2622
Utilities,2917,2,AllPub,2916
LotConfig,2919,5,Inside,2133
LandSlope,2919,3,Gtl,2778
Neighborhood,2919,25,NAmes,443
Condition1,2919,9,Norm,2511


**Interprétation :**
On retrouve 81 colonnes (en comptant `Id` et `SalePrice`) : un identifiant (`Id`), la cible
(`SalePrice`), et environ 36 variables numériques et 43 variables catégorielles décrivant le
logement (surface, qualité, équipements, localisation, année, etc.). La colonne `Id` n'a aucune
valeur prédictive : elle sera supprimée avant modélisation.


## 3. Valeurs manquantes et doublons

### 3.1 Doublons

In [11]:
nb_doublons_lignes = df.duplicated().sum()
nb_doublons_id = df['Id'].duplicated().sum()
print(f"Lignes strictement dupliquées : {nb_doublons_lignes}")
print(f"Identifiants (Id) dupliqués   : {nb_doublons_id}")


Lignes strictement dupliquées : 0
Identifiants (Id) dupliqués   : 0


**Interprétation :** aucun doublon détecté, ni sur les lignes complètes, ni sur `Id`. Chaque
ligne correspond bien à un logement unique.


### 3.2 Valeurs manquantes

In [12]:
missing = df.isna().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_summary = pd.DataFrame({'nb_manquantes': missing, 'pct_manquantes': missing_pct})
missing_summary = missing_summary[missing_summary['nb_manquantes'] > 0].sort_values('pct_manquantes', ascending=False)
missing_summary


,nb_manquantes,pct_manquantes
PoolQC,2909,99.66
MiscFeature,2814,96.40
Alley,2721,93.22
Fence,2348,80.44
MasVnrType,1766,60.50
FireplaceQu,1420,48.65
LotFrontage,486,16.65
GarageFinish,159,5.45
GarageQual,159,5.45
GarageCond,159,5.45


## 4. Incohérences et valeurs aberrantes

On vérifie ici la cohérence logique de certaines variables (dates, surfaces) et on repère les
valeurs extrêmes susceptibles d'être des erreurs de saisie ou des cas très atypiques.


In [16]:
# 4.1 Cohérence des dates : une maison ne peut pas être vendue avant sa construction,
# ni rénovée avant sa construction, ni son garage construit après l'année de vente.
incoherences = pd.DataFrame({
    "YearBuilt > YrSold": (df['YearBuilt'] > df['YrSold']).sum(),
    "YearRemodAdd < YearBuilt": (df['YearRemodAdd'] < df['YearBuilt']).sum(),
    "GarageYrBlt > YrSold": (df['GarageYrBlt'] > df['YrSold']).sum(),
}, index=["nb_lignes"]).T
incoherences


,nb_lignes
YearBuilt > YrSold,1
YearRemodAdd < YearBuilt,1
GarageYrBlt > YrSold,2


In [14]:
# Valeur aberrante connue sur ce dataset : GarageYrBlt = 2207 (erreur de saisie, la maison a été
# construite en 2006). On l'identifie explicitement.
df.loc[df['GarageYrBlt'] > df['YrSold'], ['Id', 'YearBuilt', 'GarageYrBlt', 'YrSold']]


,Id,YearBuilt,GarageYrBlt,YrSold
2549,2550,2008,2008.0,2007
2592,2593,2006,2207.0,2007


In [ ]:
# Détection d'outliers classiques : grandes surfaces habitables vendues à bas prix
outliers_grlivarea = df[(df['GrLivArea'] > 4000) & (df['SalePrice'] < 300000)]
print(f"Nombre d'outliers suspects (GrLivArea > 4000 et SalePrice < 300000) : {len(outliers_grlivarea)}")
outliers_grlivarea[['Id', 'GrLivArea', 'OverallQual', 'YearBuilt', 'SalePrice']]


Nombre d'outliers suspects (GrLivArea > 4000 et SalePrice < 300000) : 3


,Id,GrLivArea,OverallQual,YearBuilt,SalePrice
523,524,4676,10,2007,184750.000000
1298,1299,5642,10,2008,160000.000000
2549,2550,5095,10,2008,230841.338626


In [ ]:
# 4.3 Vérification des valeurs "impossibles" (surfaces négatives, années futures, etc.)
checks = pd.DataFrame({
    "Surfaces négatives (min < 0)": (df.select_dtypes(include=np.number).lt(0).sum(axis=0) > 0).sum(),
    "YrSold hors plage [2006-2010]": (~df['YrSold'].between(2006, 2010)).sum(),
    "MoSold hors plage [1-12]": (~df['MoSold'].between(1, 12)).sum(),
}, index=["résultat"]).T
checks


,résultat
Surfaces négatives (min < 0),0
YrSold hors plage [2006-2010],0
MoSold hors plage [1-12],0


## 5. Variables numériques vs catégorielles

Certaines colonnes sont numériques dans le fichier mais représentent en réalité des **catégories**
(ex : `MSSubClass` est un code de type de logement, pas une quantité). On les reclasse donc
manuellement en s'appuyant sur `data_description.txt`.


In [15]:
# Variables numériques mais sémantiquement catégorielles (codes)
numeric_but_categorical = ['MSSubClass']

num_cols = df.select_dtypes(include=np.number).columns.tolist()
cat_cols = df.select_dtypes(include='object').columns.tolist()

# Reclassement
for c in numeric_but_categorical:
    if c in num_cols:
        num_cols.remove(c)
    if c not in cat_cols:
        cat_cols.append(c)

# La cible et l'identifiant ne sont ni features numériques ni catégorielles à traiter comme telles
for c in ['Id', 'SalePrice']:
    if c in num_cols:
        num_cols.remove(c)

print(f"Nombre de variables numériques   : {len(num_cols)}")
print(f"Nombre de variables catégorielles : {len(cat_cols)}")
print("\nNumériques :", num_cols)
print("\nCatégorielles :", cat_cols)


Nombre de variables numériques   : 35
Nombre de variables catégorielles : 44

Numériques : ['LotFrontage', 'LotArea', 'OverallQual', 'OverallCond', 'YearBuilt', 'YearRemodAdd', 'MasVnrArea', 'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', '1stFlrSF', '2ndFlrSF', 'LowQualFinSF', 'GrLivArea', 'BsmtFullBath', 'BsmtHalfBath', 'FullBath', 'HalfBath', 'BedroomAbvGr', 'KitchenAbvGr', 'TotRmsAbvGrd', 'Fireplaces', 'GarageYrBlt', 'GarageCars', 'GarageArea', 'WoodDeckSF', 'OpenPorchSF', 'EnclosedPorch', '3SsnPorch', 'ScreenPorch', 'PoolArea', 'MiscVal', 'MoSold', 'YrSold']

Catégorielles : ['MSZoning', 'Street', 'Alley', 'LotShape', 'LandContour', 'Utilities', 'LotConfig', 'LandSlope', 'Neighborhood', 'Condition1', 'Condition2', 'BldgType', 'HouseStyle', 'RoofStyle', 'RoofMatl', 'Exterior1st', 'Exterior2nd', 'MasVnrType', 'ExterQual', 'ExterCond', 'Foundation', 'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2', 'Heating', 'HeatingQC', 'CentralAir', 'Electrical', 'Kit

/tmp/ipykernel_142/1139961002.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df.select_dtypes(include='object').columns.tolist()


## 6. Analyse de la variable cible `SalePrice`

In [16]:
print(df['SalePrice'].describe())
print(f"\nSkewness (asymétrie) : {df['SalePrice'].skew():.3f}")


count      2919.000000
mean     180052.854647
std       57381.565721
min       34900.000000
25%      154795.084126
50%      176734.841494
75%      191895.744157
max      755000.000000
Name: SalePrice, dtype: float64

Skewness (asymétrie) : 2.549


## 7. Séparation train / test

La séparation est effectuée **avant** tout préprocessing pour éviter la fuite de données.
Les lignes sans `SalePrice` (issues du fichier de test Kaggle) sont exclues de l'entraînement.
La cible est transformée en log pour corriger l'asymétrie (skewness ≈ 2.5).


In [ ]:
from sklearn.model_selection import train_test_split

# Correction de l'erreur de saisie connue
df.loc[df['GarageYrBlt'] == 2207, 'GarageYrBlt'] = 2007

# Suppression de Id (non prédictif)
df = df.drop(columns=['Id'])

# Suppression des 2 outliers GrLivArea > 4000 avec SalePrice connu et anormalement bas
df = df[~((df['GrLivArea'] > 4000) & (df['SalePrice'] < 300000) & df['SalePrice'].notna())]

# Séparation lignes avec / sans cible
df_train_full = df[df['SalePrice'].notna()].copy()

X = df_train_full.drop(columns=['SalePrice'])
y = np.log1p(df_train_full['SalePrice'])  # transformation log pour corriger la skewness

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"X_train : {X_train.shape}, X_val : {X_val.shape}")


## 8. Pipeline de préprocessing

Le pipeline est construit sur les données d'entraînement uniquement (`fit` sur `X_train`),
puis appliqué à `X_val` via `transform` — aucune fuite de données.

**Stratégie d'imputation :**
- Variables catégorielles : NaN = absence de la caractéristique → rempli par `'None'`
- `LotFrontage` : médiane par quartier (`Neighborhood`)
- Autres numériques : médiane

**Encodage :** `OrdinalEncoder` pour les variables ordinales connues, `OneHotEncoder` pour les nominales.

**Scaling :** `StandardScaler` sur les variables numériques.


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, OneHotEncoder
from sklearn.impute import SimpleImputer

# --- Définition des colonnes ---
numeric_but_categorical = ['MSSubClass']
num_cols = X_train.select_dtypes(include=np.number).columns.tolist()
cat_cols = X_train.select_dtypes(include='object').columns.tolist() + numeric_but_categorical
num_cols = [c for c in num_cols if c not in numeric_but_categorical]

# Variables ordinales avec ordre connu
ordinal_map = {
    'ExterQual':   ['Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'ExterCond':   ['Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'BsmtQual':    ['None', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'BsmtCond':    ['None', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'BsmtExposure':['None', 'No', 'Mn', 'Av', 'Gd'],
    'BsmtFinType1':['None', 'Unf', 'LwQ', 'Rec', 'BLQ', 'ALQ', 'GLQ'],
    'BsmtFinType2':['None', 'Unf', 'LwQ', 'Rec', 'BLQ', 'ALQ', 'GLQ'],
    'HeatingQC':   ['Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'KitchenQual': ['Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'FireplaceQu': ['None', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'GarageFinish':['None', 'Unf', 'RFn', 'Fin'],
    'GarageQual':  ['None', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'GarageCond':  ['None', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'PoolQC':      ['None', 'Fa', 'TA', 'Gd', 'Ex'],
    'Fence':       ['None', 'MnWw', 'GdWo', 'MnPrv', 'GdPrv'],
    'Functional':  ['Sal', 'Sev', 'Maj2', 'Maj1', 'Mod', 'Min2', 'Min1', 'Typ'],
}
ordinal_cols = list(ordinal_map.keys())
nominal_cols = [c for c in cat_cols if c not in ordinal_cols]

# --- Pipelines par type ---
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])

ordinal_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='None')),
    ('encoder', OrdinalEncoder(
        categories=[ordinal_map[c] for c in ordinal_cols],
        handle_unknown='use_encoded_value', unknown_value=-1
    )),
])

nominal_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='None')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
])

preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_cols),
    ('ord', ordinal_pipeline, ordinal_cols),
    ('nom', nominal_pipeline, nominal_cols),
], remainder='drop')

X_train_processed = preprocessor.fit_transform(X_train)
X_val_processed   = preprocessor.transform(X_val)
print(f"X_train_processed : {X_train_processed.shape}")
print(f"X_val_processed   : {X_val_processed.shape}")


## 9. Sauvegarde des données traitées

In [ ]:
import joblib, os

os.makedirs('../data/processed', exist_ok=True)
os.makedirs('../models', exist_ok=True)

# Récupération des noms de features après transformation
ohe_feature_names = preprocessor.named_transformers_['nom']['encoder'].get_feature_names_out(nominal_cols).tolist()
feature_names = num_cols + ordinal_cols + ohe_feature_names

pd.DataFrame(X_train_processed, columns=feature_names).assign(SalePrice=y_train.values).to_csv('../data/processed/train_processed.csv', index=False)
pd.DataFrame(X_val_processed,   columns=feature_names).assign(SalePrice=y_val.values).to_csv('../data/processed/val_processed.csv',   index=False)

joblib.dump(preprocessor, '../models/preprocessor.joblib')
print('Sauvegarde terminée.')
